# 05 - Multi-Model Router: Intelligent Request Routing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajvermatx/careeralign.com/blob/main/genai-arch/notebooks/05-multi-model-router.ipynb)

## Learning Objectives

By the end of this notebook, you will be able to:

1. Classify query complexity to determine model requirements
2. Build routing logic that maps complexity to models (cheap vs powerful)
3. Track costs per request and across the system
4. Measure and compare latency across models
5. Implement fallback chains for reliability
6. Run A/B comparisons to validate routing decisions

---

**Architecture Pattern:** Not all queries need the most powerful (and expensive) model. A router analyzes each request and sends simple queries to fast/cheap models (gpt-4o-mini) and complex queries to powerful models (gpt-4o). This can cut costs by 50-80% with minimal quality loss.

## Setup

Install the required packages.

In [ ]:
!pip install openai --quiet

## Configuration

In [ ]:
import os
import json
import time
import re
import random
from dataclasses import dataclass, field
from openai import OpenAI

api_key = os.environ.get("OPENAI_API_KEY", "")
if not api_key:
    print("WARNING: OPENAI_API_KEY not set. Mock responses will be used.")
    print("To use the real API, set your key: export OPENAI_API_KEY='sk-...'")
else:
    print("API key found. Using live OpenAI API.")

client = OpenAI(api_key=api_key if api_key else "sk-mock-key")

# Model configuration
MODELS = {
    "gpt-4o-mini": {
        "name": "gpt-4o-mini",
        "tier": "fast",
        "cost_input_per_1M": 0.15,
        "cost_output_per_1M": 0.60,
        "context_window": 128000,
        "avg_latency_ms": 400,
        "description": "Fast, cheap -- good for simple tasks"
    },
    "gpt-4o": {
        "name": "gpt-4o",
        "tier": "powerful",
        "cost_input_per_1M": 2.50,
        "cost_output_per_1M": 10.00,
        "context_window": 128000,
        "avg_latency_ms": 800,
        "description": "Powerful, more expensive -- for complex reasoning"
    }
}

print("\nModel Configuration:")
print(f"{'Model':<15} {'Tier':<10} {'Input $/1M':>12} {'Output $/1M':>12} {'Latency':>10}")
print("-" * 62)
for m in MODELS.values():
    print(f"{m['name']:<15} {m['tier']:<10} ${m['cost_input_per_1M']:>10.2f} ${m['cost_output_per_1M']:>10.2f} {m['avg_latency_ms']:>8}ms")

print(f"\nCost ratio (gpt-4o / gpt-4o-mini): {MODELS['gpt-4o']['cost_output_per_1M'] / MODELS['gpt-4o-mini']['cost_output_per_1M']:.0f}x")

---
## 1. Complexity Classifier

The classifier analyzes a query and determines its complexity level. We implement two approaches:
- **Heuristic** (fast, free): keyword matching and length analysis
- **LLM-based** (more accurate, costs a small API call): use the cheap model to classify

In [ ]:
def classify_heuristic(query):
    """Rule-based complexity classifier using heuristics."""
    query_lower = query.lower()
    word_count = len(query.split())

    # Complex indicators
    complex_patterns = [
        r"\banalyze\b", r"\bcompare\b", r"\bexplain .* in detail\b",
        r"\bwrite .* code\b", r"\bimplement\b", r"\bdebug\b",
        r"\brefactor\b", r"\barchitect\b", r"\bdesign\b",
        r"\btrade-?offs?\b", r"\bpros and cons\b", r"\bstep.by.step\b",
        r"\boptimize\b", r"\bmulti.step\b", r"\breason\b",
    ]

    # Simple indicators
    simple_patterns = [
        r"\bwhat is\b", r"\bdefine\b", r"\blist\b",
        r"\btranslate\b", r"\bsummarize\b", r"\bconvert\b",
        r"\bformat\b", r"\bhello\b", r"\bthanks?\b",
        r"\byes\b", r"\bno\b",
    ]

    complex_score = sum(1 for p in complex_patterns if re.search(p, query_lower))
    simple_score = sum(1 for p in simple_patterns if re.search(p, query_lower))

    # Long queries with multiple sentences tend to be complex
    if word_count > 50:
        complex_score += 2
    elif word_count > 25:
        complex_score += 1

    # Code blocks suggest complexity
    if "```" in query or "def " in query or "class " in query:
        complex_score += 2

    if complex_score > simple_score:
        return "complex"
    elif word_count < 10 and simple_score >= complex_score:
        return "simple"
    else:
        return "medium"


def classify_llm(query):
    """LLM-based complexity classifier (more accurate but adds latency + cost)."""
    prompt = (
        "Classify the complexity of this user query as exactly one of: simple, medium, complex.\n\n"
        "simple = factual questions, definitions, greetings, simple formatting\n"
        "medium = explanations, summaries, moderate analysis\n"
        "complex = multi-step reasoning, code generation, detailed comparisons, debugging\n\n"
        f"Query: {query}\n\n"
        'Respond with JSON: {"complexity": "simple|medium|complex", "reason": "brief explanation"}'
    )

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=60,
            temperature=0.0,
            response_format={"type": "json_object"}
        )
        result = json.loads(response.choices[0].message.content)
        return result.get("complexity", "medium")
    except Exception as e:
        print(f"API call failed: {e}")
        result = classify_heuristic(query)
        print("Using mock response for demonstration.")
        return result


# Test both classifiers
test_queries = [
    "What is Python?",
    "List the planets in our solar system.",
    "Explain the difference between TCP and UDP with examples.",
    "Write a Python class that implements a thread-safe LRU cache with TTL. Include type hints.",
    "Debug this code: def factorial(n): return n * factorial(n-1)",
    "Compare trade-offs between microservices and monolithic architecture for a startup.",
    "Hello!",
    "Summarize this article in 3 bullet points.",
]

print(f"{'Query':<70} {'Heuristic':<12} {'LLM':<10}")
print("-" * 95)
for q in test_queries:
    h = classify_heuristic(q)
    l = classify_llm(q)
    display = q[:68] + (".." if len(q) > 68 else "")
    print(f"{display:<70} {h:<12} {l:<10}")

---
## 2. Routing Logic

The router maps complexity levels to model selections. Simple and medium queries go to the cheap model; complex queries go to the powerful model.

| Complexity | Model | Rationale |
|---|---|---|
| Simple | gpt-4o-mini | Fast, cheap, sufficient quality |
| Medium | gpt-4o-mini | Good enough, saves cost |
| Complex | gpt-4o | Needs stronger reasoning |

In [ ]:
@dataclass
class RoutingDecision:
    query: str
    complexity: str
    model: str
    response: str = ""
    latency_ms: float = 0.0
    prompt_tokens: int = 0
    completion_tokens: int = 0
    cost_usd: float = 0.0
    fallback_used: bool = False


class ModelRouter:
    """Routes queries to appropriate models based on complexity."""

    def __init__(self, client, use_llm_classifier=False):
        self.client = client
        self.use_llm_classifier = use_llm_classifier
        self.routing_table = {
            "simple": "gpt-4o-mini",
            "medium": "gpt-4o-mini",
            "complex": "gpt-4o",
        }
        self.history = []

    def classify(self, query):
        if self.use_llm_classifier:
            return classify_llm(query)
        return classify_heuristic(query)

    def _calculate_cost(self, model, prompt_tokens, completion_tokens):
        config = MODELS[model]
        return (
            (prompt_tokens / 1_000_000 * config["cost_input_per_1M"]) +
            (completion_tokens / 1_000_000 * config["cost_output_per_1M"])
        )

    def route(self, query, system_prompt="You are a helpful assistant."):
        """Classify, route, and execute a query."""
        complexity = self.classify(query)
        model = self.routing_table[complexity]

        start = time.time()
        decision = RoutingDecision(query=query, complexity=complexity, model=model)

        try:
            response = self.client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": query}
                ],
                max_tokens=300
            )
            decision.response = response.choices[0].message.content
            decision.prompt_tokens = response.usage.prompt_tokens
            decision.completion_tokens = response.usage.completion_tokens
        except Exception as e:
            print(f"API call failed: {e}")
            decision.response = f"Mock: Response for '{query[:50]}...' via {model}."
            decision.prompt_tokens = len(query.split()) * 2
            decision.completion_tokens = 50
            print("Using mock response for demonstration.")

        decision.latency_ms = (time.time() - start) * 1000
        decision.cost_usd = self._calculate_cost(model, decision.prompt_tokens, decision.completion_tokens)

        self.history.append(decision)
        return decision


# Demo: route several queries
router = ModelRouter(client, use_llm_classifier=False)

queries = [
    "What is Python?",
    "Explain how gradient descent works.",
    "Write a Python class that implements a thread-safe LRU cache with TTL expiration.",
    "List 5 popular databases.",
    "Compare the trade-offs between SQL and NoSQL for a high-traffic e-commerce platform.",
]

print(f"{'Query':<55} {'Complexity':<12} {'Model':<15} {'Latency':>8} {'Cost':>10}")
print("-" * 105)

for q in queries:
    d = router.route(q)
    display = q[:53] + (".." if len(q) > 53 else "")
    print(f"{display:<55} {d.complexity:<12} {d.model:<15} {d.latency_ms:>6.0f}ms ${d.cost_usd:>8.6f}")

---
## 3. Cost Tracking

Track cumulative costs and compare against the baseline of sending everything to the most powerful model.

In [ ]:
class CostTracker:
    """Analyze routing costs and compute savings."""

    def __init__(self, history):
        self.history = history

    def total_cost(self):
        return sum(d.cost_usd for d in self.history)

    def cost_by_model(self):
        costs = {}
        for d in self.history:
            costs[d.model] = costs.get(d.model, 0) + d.cost_usd
        return costs

    def count_by_model(self):
        counts = {}
        for d in self.history:
            counts[d.model] = counts.get(d.model, 0) + 1
        return counts

    def baseline_cost(self, baseline_model="gpt-4o"):
        """What would it cost if ALL queries went to the baseline model?"""
        config = MODELS[baseline_model]
        total = 0
        for d in self.history:
            total += (
                (d.prompt_tokens / 1_000_000 * config["cost_input_per_1M"]) +
                (d.completion_tokens / 1_000_000 * config["cost_output_per_1M"])
            )
        return total

    def savings_report(self):
        actual = self.total_cost()
        baseline = self.baseline_cost()
        saved = baseline - actual
        pct = (saved / baseline * 100) if baseline > 0 else 0
        return {"actual": actual, "baseline": baseline, "savings": saved, "pct": pct}


tracker = CostTracker(router.history)

print("=== Cost Analysis ===")
print(f"\nTotal requests: {len(router.history)}")
print(f"\nRequests by model:")
for model, count in tracker.count_by_model().items():
    print(f"  {model}: {count}")

print(f"\nCost by model:")
for model, cost in tracker.cost_by_model().items():
    print(f"  {model}: ${cost:.6f}")

report = tracker.savings_report()
print(f"\n--- Savings Report ---")
print(f"Routed cost:   ${report['actual']:.6f}")
print(f"Baseline cost: ${report['baseline']:.6f} (all gpt-4o)")
print(f"Savings:       ${report['savings']:.6f} ({report['pct']:.1f}%)")

In [ ]:
# Project monthly costs at scale
complexity_distribution = {"simple": 60, "medium": 25, "complex": 15}  # Typical percentages
avg_prompt_tokens = 150
avg_completion_tokens = 200
monthly_requests = 10_000

routing_table = {"simple": "gpt-4o-mini", "medium": "gpt-4o-mini", "complex": "gpt-4o"}

routed_cost = 0
baseline_cost = 0

for complexity, pct in complexity_distribution.items():
    count = int(monthly_requests * pct / 100)
    model = routing_table[complexity]
    config = MODELS[model]
    cost = count * (
        (avg_prompt_tokens / 1_000_000 * config["cost_input_per_1M"]) +
        (avg_completion_tokens / 1_000_000 * config["cost_output_per_1M"])
    )
    routed_cost += cost

    base_config = MODELS["gpt-4o"]
    base_cost = count * (
        (avg_prompt_tokens / 1_000_000 * base_config["cost_input_per_1M"]) +
        (avg_completion_tokens / 1_000_000 * base_config["cost_output_per_1M"])
    )
    baseline_cost += base_cost

savings_pct = ((baseline_cost - routed_cost) / baseline_cost * 100) if baseline_cost > 0 else 0

print("=== Projected Monthly Cost (10K requests) ===")
print(f"\nComplexity distribution: {complexity_distribution}")
print(f"\nWith routing:     ${routed_cost:.2f}/month")
print(f"Without routing:  ${baseline_cost:.2f}/month (all gpt-4o)")
print(f"Monthly savings:  ${baseline_cost - routed_cost:.2f} ({savings_pct:.0f}%)")

---
## 4. Latency Measurement

Faster models improve user experience. We measure and compare latency across models for the same query.

In [ ]:
def measure_latency(query, model, n_runs=3):
    """Measure average latency for a query on a specific model."""
    latencies = []

    for _ in range(n_runs):
        start = time.time()
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": query}],
                max_tokens=100
            )
            elapsed = (time.time() - start) * 1000
        except Exception as e:
            # Simulate realistic latencies
            if model == "gpt-4o-mini":
                elapsed = random.uniform(180, 350)
            else:
                elapsed = random.uniform(400, 900)

        latencies.append(elapsed)

    return {
        "model": model,
        "avg_ms": sum(latencies) / len(latencies),
        "min_ms": min(latencies),
        "max_ms": max(latencies),
    }


# Compare latency
latency_queries = [
    ("What is Python?", "simple"),
    ("Explain how neural networks learn through backpropagation.", "medium"),
]

print("=== Latency Comparison ===")
print(f"{'Query':<50} {'Model':<15} {'Avg':>8} {'Min':>8} {'Max':>8}")
print("-" * 95)

for query, complexity in latency_queries:
    for model in ["gpt-4o-mini", "gpt-4o"]:
        stats = measure_latency(query, model, n_runs=2)
        display = query[:48] + (".." if len(query) > 48 else "")
        print(f"{display:<50} {model:<15} {stats['avg_ms']:>6.0f}ms {stats['min_ms']:>6.0f}ms {stats['max_ms']:>6.0f}ms")

---
## 5. Fallback Chains

In production, models can fail (rate limits, outages, timeouts). A fallback chain tries models in order until one succeeds.

In [ ]:
class FallbackRouter:
    """Router with automatic fallback on model failure."""

    def __init__(self, client):
        self.client = client
        self.fallback_chains = {
            "simple":  ["gpt-4o-mini", "gpt-4o"],
            "medium":  ["gpt-4o-mini", "gpt-4o"],
            "complex": ["gpt-4o", "gpt-4o-mini"],
        }

    def route(self, query, complexity=None):
        if complexity is None:
            complexity = classify_heuristic(query)

        chain = self.fallback_chains[complexity]

        for i, model in enumerate(chain):
            try:
                start = time.time()
                response = self.client.chat.completions.create(
                    model=model,
                    messages=[{"role": "user", "content": query}],
                    max_tokens=200,
                    timeout=10
                )
                elapsed = (time.time() - start) * 1000

                return {
                    "model_used": model,
                    "response": response.choices[0].message.content,
                    "latency_ms": elapsed,
                    "fallback_used": i > 0,
                    "attempts": i + 1,
                    "complexity": complexity
                }
            except Exception as e:
                print(f"  Model {model} failed: {e}")
                if i < len(chain) - 1:
                    print(f"  Falling back to {chain[i+1]}...")

        # All models failed
        print("  All models in fallback chain failed.")
        return {
            "model_used": "mock",
            "response": f"Mock: Fallback response for '{query[:50]}...'",
            "latency_ms": 0,
            "fallback_used": True,
            "attempts": len(chain),
            "complexity": complexity
        }


# Demo
fb_router = FallbackRouter(client)

test_queries = [
    ("What is a REST API?", "simple"),
    ("Write a function to implement merge sort with detailed comments.", "complex"),
]

print("=== Fallback Chain Demo ===")
for q, complexity in test_queries:
    print(f"\nQuery: {q}")
    print(f"Complexity: {complexity}")
    print(f"Chain: {fb_router.fallback_chains[complexity]}")
    result = fb_router.route(q, complexity=complexity)
    print(f"Model used: {result['model_used']}")
    print(f"Fallback used: {result['fallback_used']}")
    print(f"Response: {result['response'][:120]}...")

---
## 6. A/B Comparison

Before deploying routing decisions, run A/B comparisons to verify that the cheaper model produces acceptable quality. Send the same query to both models and compare using an LLM-as-judge.

In [ ]:
def ab_compare(query, model_a="gpt-4o-mini", model_b="gpt-4o"):
    """Send the same query to two models and compare results."""
    results = {}

    for model in [model_a, model_b]:
        start = time.time()
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": query}],
                max_tokens=200,
                temperature=0.0
            )
            content = response.choices[0].message.content
            tokens = response.usage.total_tokens
        except Exception as e:
            print(f"API call failed: {e}")
            if model == "gpt-4o-mini":
                content = ("Mock: Python is a high-level, interpreted programming language "
                           "known for its simplicity and readability. It supports multiple "
                           "paradigms including procedural, object-oriented, and functional programming.")
            else:
                content = ("Mock: Python is a versatile, high-level programming language created "
                           "by Guido van Rossum in 1991. It emphasizes code readability with its "
                           "use of significant indentation. Python supports multiple programming "
                           "paradigms and has a comprehensive standard library, making it suitable "
                           "for web development, data science, AI/ML, automation, and more.")
            tokens = len(content.split()) * 2
            print("Using mock response for demonstration.")

        elapsed = (time.time() - start) * 1000
        config = MODELS[model]
        cost = tokens / 1_000_000 * (config["cost_input_per_1M"] + config["cost_output_per_1M"]) / 2

        results[model] = {
            "response": content,
            "latency_ms": elapsed,
            "tokens": tokens,
            "cost_usd": cost,
            "length": len(content)
        }

    return results


# A/B comparison
ab_queries = [
    "What is Python?",
    "Explain the CAP theorem in distributed systems.",
    "Write a Python function to find the longest common subsequence of two strings.",
]

print("=== A/B Model Comparison ===")
for q in ab_queries:
    print(f"\n{'='*70}")
    print(f"Query: {q}")
    print(f"{'='*70}")

    results = ab_compare(q)
    for model, data in results.items():
        print(f"\n--- {model} ---")
        print(f"Response: {data['response'][:150]}...")
        print(f"Latency: {data['latency_ms']:.0f}ms | Length: {data['length']} chars | Cost: ${data['cost_usd']:.6f}")

In [ ]:
# LLM-as-judge to score both responses
def judge_comparison(query, response_a, response_b):
    """Use LLM to judge which response is better."""
    judge_prompt = (
        f"Compare these two AI responses to the same question.\n\n"
        f"Question: {query}\n\n"
        f"Response A (gpt-4o-mini): {response_a[:500]}\n\n"
        f"Response B (gpt-4o): {response_b[:500]}\n\n"
        f"Rate each response 1-5 on: accuracy, completeness, clarity.\n"
        f'Respond as JSON: {{"response_a": {{"accuracy": N, "completeness": N, "clarity": N}}, '
        f'"response_b": {{"accuracy": N, "completeness": N, "clarity": N}}, "winner": "A|B|tie"}}'
    )

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": judge_prompt}],
            max_tokens=200,
            temperature=0.0,
            response_format={"type": "json_object"}
        )
        result = json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"API call failed: {e}")
        result = {
            "response_a": {"accuracy": 4, "completeness": 4, "clarity": 5},
            "response_b": {"accuracy": 5, "completeness": 5, "clarity": 5},
            "winner": "tie"
        }
        print("Using mock response for demonstration.")

    return result


# Judge each A/B comparison
print("=== LLM-as-Judge Evaluation ===")
print(f"{'Query':<50} {'A (mini)':>10} {'B (4o)':>10} {'Winner':>8}")
print("-" * 82)

for q in ab_queries:
    ab = ab_compare(q)
    models_list = list(ab.keys())
    judgment = judge_comparison(q, ab[models_list[0]]["response"], ab[models_list[1]]["response"])

    a_scores = judgment.get("response_a", {})
    b_scores = judgment.get("response_b", {})
    a_avg = sum(a_scores.values()) / max(len(a_scores), 1)
    b_avg = sum(b_scores.values()) / max(len(b_scores), 1)
    winner = judgment.get("winner", "tie")

    display = q[:48] + (".." if len(q) > 48 else "")
    print(f"{display:<50} {a_avg:>8.1f}/5 {b_avg:>8.1f}/5 {winner:>8}")

print(f"\nConclusion: If scores are close, use the cheaper model to save cost.")
print(f"Only upgrade to gpt-4o when quality difference is significant.")

---
## Summary

This notebook built a multi-model router for cost-efficient LLM usage:

| Component | What It Does | Key Insight |
|---|---|---|
| Complexity Classifier | Analyzes query difficulty | Heuristics are fast; LLM is more accurate |
| Routing Logic | Maps complexity to model | Simple/medium -> cheap; complex -> powerful |
| Cost Tracking | Monitors per-request and cumulative costs | Routing can save 50-80% vs always using gpt-4o |
| Latency Measurement | Compares response times across models | Cheaper models are typically 2-3x faster |
| Fallback Chains | Tries backup models on failure | Essential for production reliability |
| A/B Comparison | Validates routing decisions | LLM-as-judge confirms quality is acceptable |

**Key takeaways:**
- Most queries (60-80%) are simple enough for the cheapest model
- Routing saves significant cost with minimal quality loss
- Always validate routing decisions with A/B testing before production deployment
- Fallback chains prevent outages from model-specific failures
- Track costs continuously to catch regressions